In [1]:
"""
Training and evaluation loop for the U-Net DQ estimator.

Features:
  - Combined MSE+MAE loss
  - R², RMSE, MAE tracked every epoch
  - Learning-rate scheduler (ReduceLROnPlateau on val RMSE)
  - Best-model checkpointing (lowest val RMSE)
  - Early stopping
  - Full training history saved to CSV
"""

import time
import csv
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from unet_model           import UNet
from attention_unet_model import AttentionUNet
from dataset               import get_dataloaders
from metrics               import CombinedMSEMAELoss, compute_metrics


# ---------------------------------------------------------------------------
# Config  (edit here or import from a separate config file)
# ---------------------------------------------------------------------------

CONFIG = {
    # Data
    "train_path":     "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/Vmag_prj/V_MAG_dataset/case118/dataset_train.npz",
    "val_path":       "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/Vmag_prj/V_MAG_dataset/case118/dataset_val.npz",
    "test_path":      "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/Vmag_prj/V_MAG_dataset/case118/dataset_test.npz",

    # Task
    "target_key":      "Vmag",     # "y" for DQ matrix target, "Vmag" for per-bus voltage vector target
    "output_mode":     "vector",  # "matrix" (DQ) or "vector" (Vmag) - must match target_key's shape

    # Model
    "model_type":      "attention_unet",  # "unet" or "attention_unet"
    "base_channels":   32,                # channel multiplier
    "attn_heads":      4,                 # bottleneck self-attention heads (attention_unet only)

    # Training
    "epochs":         100,
    "batch_size":     64,
    "learning_rate":  1e-3,
    "weight_decay":   1e-4,        # L2 regularization
    "loss_alpha":     0.5,         # weight of MSE vs MAE  (0.5 = equal)

    # Scheduler
    "lr_patience":    10,          # epochs without improvement before LR drop
    "lr_factor":      0.5,         # LR multiplied by this on plateau

    # Early stopping
    "early_stop_patience": 45,     # epochs without improvement before stop

    # Output  (set to None to auto-name based on model_type)
    "checkpoint_path": None,
    "history_path":    None,

    # Hardware
    "num_workers":    0,           # set >0 on Linux for faster loading
}


# ---------------------------------------------------------------------------
# Single-epoch helpers
# ---------------------------------------------------------------------------

def train_one_epoch(
    model:      nn.Module,
    loader:     DataLoader,
    criterion:  nn.Module,
    optimizer:  torch.optim.Optimizer,
    device:     torch.device,
) -> dict[str, float]:
    model.train()
    total_loss = 0.0
    all_pred, all_target = [], []

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        all_pred.append(pred.detach().cpu())
        all_target.append(y.detach().cpu())

    all_pred   = torch.cat(all_pred)
    all_target = torch.cat(all_target)
    metrics    = compute_metrics(all_pred, all_target)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    all_pred, all_target = [], []

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)

        total_loss += loss.item() * X.size(0)
        all_pred.append(pred.cpu())
        all_target.append(y.cpu())

    all_pred   = torch.cat(all_pred)
    all_target = torch.cat(all_target)
    metrics    = compute_metrics(all_pred, all_target)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics


# ---------------------------------------------------------------------------
# Main training loop
# ---------------------------------------------------------------------------

def train(cfg: dict = CONFIG):
    cfg = dict(cfg)  # avoid mutating the module-level CONFIG
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Model type: {cfg['model_type']}\n")

    # Auto-name outputs per model type if not explicitly set
    if cfg["checkpoint_path"] is None:
        cfg["checkpoint_path"] = f"best_model_{cfg['model_type']}_{cfg['target_key']}_case118.pth"
    if cfg["history_path"] is None:
        cfg["history_path"] = f"training_history_{cfg['model_type']}_{cfg['target_key']}_case118.csv"

    # Data
    train_loader, val_loader, test_loader = get_dataloaders(
        train_path  = cfg["train_path"],
        val_path    = cfg["val_path"],
        test_path   = cfg["test_path"],
        batch_size  = cfg["batch_size"],
        num_workers = cfg["num_workers"],
        target_key  = cfg["target_key"],
    )

    # Model
    if cfg["model_type"] == "unet":
        model = UNet(
            base_channels=cfg["base_channels"],
            output_mode=cfg["output_mode"],
        ).to(device)
    elif cfg["model_type"] == "attention_unet":
        model = AttentionUNet(
            base_channels=cfg["base_channels"],
            attn_heads=cfg["attn_heads"],
            output_mode=cfg["output_mode"],
        ).to(device)
    else:
        raise ValueError(f"Unknown model_type: {cfg['model_type']!r}. Use 'unet' or 'attention_unet'.")

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {total_params:,}\n")

    # Loss, optimizer, scheduler
    criterion = CombinedMSEMAELoss(alpha=cfg["loss_alpha"])
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = cfg["learning_rate"],
        weight_decay = cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        patience  = cfg["lr_patience"],
        factor    = cfg["lr_factor"],
        #verbose   = True,
    )

    # History CSV
    history_path = Path(cfg["history_path"])
    csv_file  = history_path.open('w', newline='')
    csv_writer = csv.DictWriter(csv_file, fieldnames=[
        'epoch', 'lr',
        'train_loss', 'train_r2', 'train_rmse', 'train_mae',
        'val_loss',   'val_r2',   'val_rmse',   'val_mae',
    ])
    csv_writer.writeheader()

    # Training state
    best_val_rmse    = float('inf')
    epochs_no_improve = 0

    print(f"{'Epoch':>6} | {'LR':>8} | {'Train Loss':>10} {'Train R²':>9} {'Train RMSE':>10} {'Train MAE':>9}"
          f" | {'Val Loss':>9} {'Val R²':>8} {'Val RMSE':>9} {'Val MAE':>8}")
    print("-" * 110)

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_m   = evaluate(model, val_loader, criterion, device)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_m['rmse'])

        # Logging
        print(
            f"{epoch:>6} | {current_lr:>8.2e} |"
            f" {train_m['loss']:>10.5f} {train_m['r2']:>9.4f} {train_m['rmse']:>10.5f} {train_m['mae']:>9.5f}"
            f" | {val_m['loss']:>9.5f} {val_m['r2']:>8.4f} {val_m['rmse']:>9.5f} {val_m['mae']:>8.5f}"
            f"  [{time.time()-t0:.1f}s]"
        )

        csv_writer.writerow({
            'epoch':      epoch,
            'lr':         current_lr,
            'train_loss': train_m['loss'], 'train_r2': train_m['r2'],
            'train_rmse': train_m['rmse'], 'train_mae': train_m['mae'],
            'val_loss':   val_m['loss'],   'val_r2':   val_m['r2'],
            'val_rmse':   val_m['rmse'],   'val_mae':  val_m['mae'],
        })
        csv_file.flush()

        # Checkpointing
        if val_m['rmse'] < best_val_rmse:
            best_val_rmse = val_m['rmse']
            epochs_no_improve = 0
            torch.save({
                'epoch':      epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_rmse':   best_val_rmse,
                'val_r2':     val_m['r2'],
                'val_mae':    val_m['mae'],
                'config':     cfg,
            }, cfg["checkpoint_path"])
            print(f"         ↳ New best val RMSE: {best_val_rmse:.5f}  → saved checkpoint")
        else:
            epochs_no_improve += 1

        # Early stopping
        if epochs_no_improve >= cfg["early_stop_patience"]:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg['early_stop_patience']} epochs).")
            break

    csv_file.close()
    print(f"\nTraining complete. Best val RMSE: {best_val_rmse:.5f}")
    print(f"Checkpoint saved to : {cfg['checkpoint_path']}")
    print(f"History saved to    : {cfg['history_path']}")

    # ------------------------------------------------------------------
    # Final test evaluation with best checkpoint
    # ------------------------------------------------------------------
    print("\n--- Test Evaluation (best checkpoint) ---")
    checkpoint = torch.load(cfg["checkpoint_path"], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    test_m = evaluate(model, test_loader, criterion, device)

    print(f"  R²  : {test_m['r2']:.4f}")
    print(f"  RMSE: {test_m['rmse']:.6f}")
    print(f"  MAE : {test_m['mae']:.6f}")

    return model, test_m


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    train(CONFIG)


Device: cuda
Model type: attention_unet

Train : DependencyMatrixDataset(n=7734, target_key='Vmag', X=(7734, 1, 118, 118), y=(7734, 118))
Val   : DependencyMatrixDataset(n=1657, target_key='Vmag', X=(1657, 1, 118, 118), y=(1657, 118))
Test  : DependencyMatrixDataset(n=1659, target_key='Vmag', X=(1659, 1, 118, 118), y=(1659, 118))
Model parameters: 2,196,323

 Epoch |       LR | Train Loss  Train R² Train RMSE Train MAE |  Val Loss   Val R²  Val RMSE  Val MAE
--------------------------------------------------------------------------------------------------------------
     1 | 1.00e-03 |    0.04741  -59.4734    0.17912   0.06273 |   0.00586   0.5978   0.01461  0.01150  [12.3s]
         ↳ New best val RMSE: 0.01461  → saved checkpoint
     2 | 1.00e-03 |    0.00306    0.8901    0.00764   0.00606 |   0.00881   0.2594   0.01982  0.01722  [12.0s]
     3 | 1.00e-03 |    0.00264    0.9272    0.00622   0.00524 |   0.00561   0.6704   0.01322  0.01105  [12.0s]
         ↳ New best val RMSE: 0.013